**⭐ 1. What This Pattern Solves**

Extracts deeply nested JSON into a flat tabular structure.
Makes semi-structured API, log, or event data SQL/query-friendly.
Simplifies aggregations, joins, and analytics on nested fields.
Handles arrays, dictionaries, and mixed-type nested structures in pipelines.

**⭐ 2. SQL Equivalent**

In [0]:
%sql
-- Flatten JSON arrays or nested objects
SELECT 
    j.id,
    j.name,
    a.value AS item_value
FROM json_table jt
JOIN LATERAL UNNEST(jt.items) AS a(value) ON TRUE;

**⭐ 3. Core Idea**

Recursively traverse JSON tree.
Map nested keys into a single flat key path.

**⭐ 4. Template Code (MEMORIZE THIS)**

In [0]:
def flatten_json(d, parent_key='', sep='.'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_json(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            for i, item in enumerate(v):
                if isinstance(item, dict):
                    items.extend(flatten_json(item, f"{new_key}[{i}]", sep=sep).items())
                else:
                    items.append((f"{new_key}[{i}]", item))
        else:
            items.append((new_key, v))
    return dict(items)

**⭐ 5. Detailed Example**

In [0]:
data = {
    "user": {"id": 1, "name": "Alice"},
    "events": [{"type": "click", "time": "10:00"}, {"type": "view", "time": "10:05"}]
}

flattened = flatten_json(data)
print(flattened)

{
    'user.id': 1,
    'user.name': 'Alice',
    'events[0].type': 'click',
    'events[0].time': '10:00',
    'events[1].type': 'view',
    'events[1].time': '10:05'
}

**⭐ 6. Mini Practice Problems**

Flatten this JSON: {"order": {"id": 101, "items": [{"sku": "A1", "qty": 2}, {"sku": "B2", "qty": 3}]}}
Flatten a nested JSON with arrays inside dictionaries inside arrays.

**⭐ 7. Full Data Engineering Scenario**

Problem Statement: API logs provide nested JSON with user, session, and events. Your pipeline needs to store flattened rows in a warehouse for analytics.

Expected Output: Table with columns like:
user.id, user.name, session.id, events[0].type, events[0].time, events[1].type, events[1].time

Skeleton Solution:

In [0]:
flattened_rows = [flatten_json(record) for record in api_logs]
# Convert to DataFrame or write to warehouse

**⭐ 8. Time & Space Complexity**

Time Complexity: O(n) — n = total keys/elements in JSON
Space Complexity: O(n) — each key/value is stored in flat dict

**⭐ 9. Common Pitfalls & Mistakes**

❌ Not handling lists properly → loses array items
❌ Using mutable default args in recursion → unexpected behavior
✔ Correct approach: recursively handle dicts and lists separately
✔ Keep key paths unique with indices for arrays
✔ Avoid deep recursion on very large JSON; consider iterative stack-based flattening